In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/dysarthria-data'))

In [ ]:
!git clone https://github.com/paloma888/dysarthria-asr-personalization.git

In [ ]:
!pip install transformers datasets peft accelerate evaluate jiwer soundfile librosa torchcodec -q

In [ ]:
import sys
sys.path.append('/content/dysarthria-asr-personalization/training')
from pipeline import gather_torgo, gather_uaspeech

In [ ]:
!cp -r /content/drive/MyDrive/dysarthria-data /content/data

In [ ]:
!git -C /content/dysarthria-asr-personalization pull

In [ ]:
from pipeline import filter_valid_audio

In [ ]:
torgo = gather_torgo('/content/data/TORGO')
torgo = filter_valid_audio(torgo)
ua = gather_uaspeech('/content/data/UASpeech/audio', '/content/data/UASpeech/mlf')
ua = filter_valid_audio(ua)

In [ ]:
from pipeline import group_by_speaker, train_val_test_split_torgo, train_val_test_split_ua
torgo_groups = group_by_speaker(torgo)
train_torgo, val_torgo, test_torgo = train_val_test_split_torgo(torgo_groups)

train_ua, val_ua, test_ua = train_val_test_split_ua(ua)

#combined training on torgo + uaspeech for a more general base
total_train = train_torgo + train_ua
total_val = val_torgo + val_ua
total_test = test_torgo + test_ua
cont_test = sum(1 for p in total_test if not p["is_isolated"])
print("total continuous in test set: ", cont_test)

In [ ]:
print(f"total train:{len(total_train)}, total val: {len(total_val)}, total test: {len(total_test)}")

In [ ]:
import random
random.seed(20)
random.shuffle(total_train)
random.shuffle(total_val)
random.shuffle(total_test)

In [ ]:
#subset for a fast first run (temporary)
iso_test = [p for p in total_test if p["is_isolated"]]
cont_test = [p for p in total_test if not p["is_isolated"]]
small_run_train = total_train[:4000]
small_run_val = total_val[:400]
small_run_test = cont_test + iso_test[:431]
random.shuffle(small_run_test)

total_iso_test = sum(1 for p in small_run_test if p["is_isolated"])
total_cont_test = sum(1 for p in small_run_test if not p["is_isolated"])

print("iso: ", total_iso_test)
print("cont: ", total_cont_test)

total_iso_train = sum(1 for p in small_run_train if p["is_isolated"])
total_cont_train = sum(1 for p in small_run_train if not p["is_isolated"])

print("iso: ", total_iso_train)
print("cont: ", total_cont_train)

In [ ]:
print(f"subset — train: {len(small_run_train)}, val: {len(small_run_val)}, test: {len(small_run_test)}")

In [ ]:
from pipeline import dataset_from_pairs, build_training_info
#replace with total_train, total_val, and total_test later
train_ds = dataset_from_pairs(small_run_train)
val_ds = dataset_from_pairs(small_run_val)
test_ds = dataset_from_pairs(small_run_test)

train_ds = train_ds.map(build_training_info)
val_ds = val_ds.map(build_training_info)
test_ds = test_ds.map(build_training_info)

print(train_ds)
import numpy as np
print(np.shape(train_ds[0]["input_features"]))

In [ ]:
!pip install -U torchao -q

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

lora = LoraConfig(r = 16, lora_alpha = 32, target_modules = ["q_proj", "v_proj"], lora_dropout = 0.05)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from pipeline import processor
from pipeline import DataCollatorSpeechSeq2Seq
collator = DataCollatorSpeechSeq2Seq(processor)

In [ ]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
training_args = Seq2SeqTrainingArguments(
    output_dir = "/content/drive/MyDrive/checkpoints-4000",
    per_device_train_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=10,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    seed=20,
    load_best_model_at_end = True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model = model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class= processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train(resume_from_checkpoint=True)

In [ ]:
device = "cuda"
model.to(device)

In [ ]:
from eval import inference
pred = inference(model, test_ds[0]["audio"], device)
print("Predicted:", pred)
print("Actual:", test_ds[0]["text"])

In [ ]:
from eval import get_metrics
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
base_model.config.forced_decoder_ids = None
base_model.config.suppress_tokens = []


wer_cont, cer_cont, wra_iso, cer_iso = get_metrics(base_model, test_ds)
print(f"Baseline Continuous: WER: {wer_cont:.2%}, CER: {cer_cont:.2%}")
print(f"Baseline Isolated: CER: {cer_iso:.2%}, WRA: {wra_iso:.2%}")

In [ ]:
wer_cont, cer_cont, wra_iso, cer_iso = get_metrics(model, test_ds)
print(f"Fine-tuned Continuous: WER: {wer_cont:.2%}, CER: {cer_cont:.2%}")
print(f"Fine-tuned Isolated: CER: {cer_iso:.2%}, WRA: {wra_iso:.2%}")